# 03 Model Training and Evaluation

Professional People Analytics notebook for Employee Attrition Prediction & Workforce Intelligence.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "src").exists() and (candidate / "data" / "raw" / "WA_Fn-UseC_-HR-Employee-Attrition.csv").exists():
            return candidate
    raise RuntimeError("Project root not found. Open the notebook from the repository or one of its subfolders.")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

WindowsPath('D:/Project/Data Science/employee_attrition_prediction_&_workforce_intelligence')

## Objective

Compare model candidates using stratified splitting and cross-validation. The operating threshold is selected on validation data and final performance is reported on the untouched test split.

In [2]:
from src.data import load_raw_data
from src.features import build_modeling_dataset
from src.models import train_and_evaluate

X, y, featured = build_modeling_dataset(load_raw_data())
results = train_and_evaluate(X, y)
results["best_model_name"], results["selected_threshold"], results["test_metrics"]

('Logistic Regression',
 0.5,
 {'threshold': 0.5,
  'roc_auc': 0.8294426737875786,
  'pr_auc': 0.5575211292624696,
  'precision': 0.39759036144578314,
  'recall': 0.7021276595744681,
  'f1': 0.5076923076923077,
  'fbeta_2': 0.6088560885608856,
  'balanced_accuracy': 0.7498492548884486,
  'true_negative': 197,
  'false_positive': 50,
  'false_negative': 14,
  'true_positive': 33,
  'selected_employees': 83,
  'intervention_rate': 0.282312925170068})

## Model Comparison

In [3]:
results['model_comparison']

,model,cv_roc_auc_mean,cv_pr_auc_mean,cv_recall_mean,cv_balanced_accuracy_mean,validation_roc_auc,validation_pr_auc,validation_recall_at_0_50,validation_precision_at_0_50,selection_score,baseline_pr_auc,selected_model
0,Logistic Regression,0.821189,0.627856,0.700000,0.746021,0.802481,0.628553,0.744681,0.426829,0.703983,0.161565,True
2,Random Forest,0.813751,0.572861,0.410526,0.682941,0.809200,0.598124,0.446809,0.636364,0.587379,0.161565,False
3,Gradient Boosting,0.819551,0.624134,0.315789,0.652819,0.806357,0.582069,0.276596,0.812500,0.520011,0.161565,False
1,Decision Tree,0.694300,0.376019,0.568421,0.666024,0.694332,0.358305,0.617021,0.345238,0.516061,0.161565,False


## Threshold Optimization

In [4]:
results['threshold_table'].sort_values('fbeta_2', ascending=False).head(12)

,threshold,roc_auc,pr_auc,precision,recall,f1,fbeta_2,balanced_accuracy,true_negative,false_positive,false_negative,true_positive,selected_employees,intervention_rate
9,0.50,0.802481,0.628553,0.426829,0.744681,0.542636,0.648148,0.777199,200,47,12,35,82,0.278912
8,0.45,0.802481,0.628553,0.391304,0.765957,0.517986,0.642857,0.769618,191,56,11,36,92,0.312925
7,0.40,0.802481,0.628553,0.362745,0.787234,0.496644,0.637931,0.762038,182,65,10,37,102,0.346939
10,0.55,0.802481,0.628553,0.452055,0.702128,0.550000,0.632184,0.770092,207,40,14,33,73,0.248299
6,0.35,0.802481,0.628553,0.345794,0.787234,0.480519,0.627119,0.751917,177,70,10,37,107,0.363946
13,0.70,0.802481,0.628553,0.566038,0.638298,0.600000,0.622407,0.772590,224,23,17,30,53,0.180272
5,0.30,0.802481,0.628553,0.318966,0.787234,0.453988,0.608553,0.733698,168,79,10,37,116,0.394558
12,0.65,0.802481,0.628553,0.508475,0.638298,0.566038,0.607287,0.760444,218,29,17,30,59,0.200680
11,0.60,0.802481,0.628553,0.449275,0.659574,0.534483,0.603113,0.752864,209,38,16,31,69,0.234694
4,0.25,0.802481,0.628553,0.280303,0.787234,0.413408,0.578125,0.701309,152,95,10,37,132,0.448980


## Holdout Prediction Review

In [5]:
import pandas as pd

prediction_review = pd.DataFrame(
    {
        "actual_attrition_flag": results["y_test"].values,
        "attrition_probability": results["test_probability"],
        "predicted_attrition_flag": (results["test_probability"] >= results["selected_threshold"]).astype(int),
    },
    index=results["y_test"].index,
).sort_values("attrition_probability", ascending=False)

prediction_review.head(15)

,actual_attrition_flag,attrition_probability,predicted_attrition_flag
911,1,0.995374,1
688,1,0.995020,1
357,1,0.992609,1
36,1,0.961739,1
1021,1,0.957213,1
711,1,0.940896,1
347,0,0.940892,1
301,0,0.936336,1
1391,0,0.913831,1
946,1,0.909722,1
